In [ ]:
import os

# 1. 强制清理旧残留
print("正在清理旧文件...")
!rm -rf Diffusion-Illusions
!rm -rf Diffusion-Illusion # 把我之前写错的那个也删掉
!rm -rf master.zip

# 2. 克隆仓库 (注意：这次名字是对的！)
print("正在克隆仓库...")
!git clone https://github.com/RyannDaGreat/Diffusion-Illusions

# 3. 检查是否成功
if os.path.exists('Diffusion-Illusions'):
    print("✅ 仓库克隆成功！")
    
    # 4. 进入目录
    %cd Diffusion-Illusions
    
    # 5. 安装依赖
    print("正在安装依赖 (红色警告请忽略)...")
    !pip install -r requirements.txt
    !pip install mediapy easydict "numpy<2.0"
    
    print("\n✅✅ 环境初始化全部完成！")
    print("⚠️⚠️ 现在的关键步骤：请点击上方菜单 'Runtime' -> 'Restart session' 重启运行时！")
    
else:
    print("❌❌ 克隆还是失败了，请检查网络。")

In [ ]:
import os

# 定义仓库名字（注意带 's'）
repo_name = "Diffusion-Illusions"

# 检查当前是否已经在文件夹里了
if os.getcwd().endswith(repo_name):
    print(f"✅ 当前位置正确: {os.getcwd()}")
else:
    # 如果不在，就尝试进去
    if os.path.exists(repo_name):
        %cd {repo_name}
        print(f"✅ 已切换工作目录到: {os.getcwd()}")
    else:
        # 如果文件夹都不存在，说明之前的克隆没成功，重新克隆一下
        print("⚠️ 文件夹不存在，正在重新克隆...")
        !git clone https://github.com/RyannDaGreat/Diffusion-Illusions
        %cd {repo_name}
        print(f"✅ 克隆并切换完成: {os.getcwd()}")

In [ ]:
from IPython.display import clear_output
from rp import *
import numpy as np
import rp
import torch
import torch.nn as nn
import source.stable_diffusion as sd
from source.learnable_textures import LearnableImageFourier
from source.stable_diffusion_labels import NegativeLabel
from itertools import chain
import torchvision.transforms.functional as TF
import torch.nn.functional as F
import math
from google.colab import files
from PIL import Image, ImageOps

# === 核心工具函数：修正版柱面投影变换 ===

def get_anamorphic_grid(size=512, r_min=0.2, r_max=0.9):
    """
    修正后的映射：将直角坐标的目标图映射到极坐标底图上。
    r_min: 圆柱体半径
    r_max: 纸张边缘半径
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 创建目标图的坐标系 [-1, 1]
    # 我们假设人眼看到的反射图是在圆柱面上的“平铺”视图
    y, x = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing='ij')
    
    # 关键物理变换：
    # 目标图的 X 坐标对应 极坐标的角度 theta
    # 目标图的 Y 坐标对应 极坐标的半径 r
    theta = x * math.pi # 映射到 [-pi, pi]
    r = (y + 1) / 2 * (r_max - r_min) + r_min # 映射到 [r_min, r_max]
    
    # 转换回底图（纸面）需要的直角坐标用于采样
    grid_x = r * torch.cos(theta)
    grid_y = r * torch.sin(theta)
    
    # grid_sample 的输入范围必须在 [-1, 1]
    grid = torch.stack((grid_x, grid_y), dim=-1).to(device)
    return grid

def apply_cylindrical_transform(distorted_image, grid):
    """
    模拟反射过程
    """
    # 使用 bilinear 插值采样，确保梯度回传顺畅
    output = F.grid_sample(distorted_image, grid.unsqueeze(0), 
                           mode='bilinear', padding_mode='zeros', align_corners=True)
    return output

def create_annular_mask(size=512, r_min=0.2, r_max=0.9):
    y, x = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing='ij')
    r = torch.sqrt(x**2 + y**2)
    mask = (r >= r_min) & (r <= r_max)
    return mask.float().unsqueeze(0).unsqueeze(0).to(torch.device('cuda'))

# 初始化参数
SIZE = 512 
R_MIN = 0.25 
R_MAX = 0.95 
ANAMORPHIC_GRID = get_anamorphic_grid(SIZE, R_MIN, R_MAX)
ANNULAR_MASK = create_annular_mask(SIZE, R_MIN, R_MAX)

In [ ]:
if 'model_sd' not in dir():
    print("正在加载 Stable Diffusion...")
    model_name = "CompVis/stable-diffusion-v1-4"
    gpu = rp.select_torch_device()
    model_sd = sd.StableDiffusion(gpu, model_name)
    device = model_sd.device
    
    # 确保网格和遮罩在正确的设备上
    ANAMORPHIC_GRID = ANAMORPHIC_GRID.to(device)
    ANNULAR_MASK = ANNULAR_MASK.to(device)
    print("模型与网格加载完毕！")
else:
    print("模型已存在。")

In [ ]:
print(">>> 请上传你希望在圆柱反射中看到的‘最终目标’图片 (例如美女、清晰的脸) <<<")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    target_pil = Image.open(filename).convert('RGB')
    target_pil = ImageOps.fit(target_pil, (SIZE, SIZE), method=Image.Resampling.LANCZOS)
    
    # 目标图直接作为最终反射的结果
    secret_target_tensor = TF.to_tensor(target_pil).to(device).unsqueeze(0)
    
    print("\n这是你希望在圆柱反射中看到的效果：")
    rp.display_image(rp.as_numpy_image(secret_target_tensor[0]))
else:
    print("❌ 未上传图片。")

In [ ]:
# === 🎨 实验参数 ===
GUIDANCE_STRENGTH = 4000 # 变形修复强度

# 纸面上看到的图（扭曲的怪兽）
prompt_distorted = "A terrifying distorted monster face, melting textures, surreal horror, vibrant colors"
# 反射里看到的图（虽然我们直接对齐像素，但 SD 引导有助于保持质感）
prompt_target = "A beautiful elegant woman portrait, highly detailed, soft lighting"

negative_prompt = "blur, low quality, distortion, text, watermark"

# 只创建一个生成器：即打印在纸上的那张图
raw_image = LearnableImageFourier(height=SIZE, width=SIZE, hidden_dim=256, num_features=256).to(device)

def get_distorted_image():
    # 经过环形遮罩处理，中间留出放圆柱的空洞
    return raw_image() * ANNULAR_MASK

label_distorted = NegativeLabel(prompt_distorted, negative_prompt)

optim = torch.optim.SGD(raw_image.parameters(), lr=1e-4)

print("初始化完成：我们将生成一张看似怪兽的扭曲图，使其在圆柱反射中变脸。")

In [ ]:
NUM_ITER = 3000
DISPLAY_INTERVAL = 100 # 加快刷新频率观察变化

# 调整 SD 步长，让其在后期再介入，前期先对齐像素
model_sd.max_step = 900
model_sd.min_step = 100

# 提高隐写强度，如果还是看不清，可以加到 8000
GUIDANCE_STRENGTH = 6000 

display_eta = rp.eta(NUM_ITER, title='Anamorphosis Training')

try:
    for iter_num in range(NUM_ITER):
        display_eta(iter_num)

        img_distorted = get_distorted_image()

        # --- A. 语义引导 (怪兽提示词) ---
        # 减小 noise_coef，防止噪声打碎好不容易学到的像素特征
        _ = model_sd.train_step(
            label_distorted.embedding,
            img_distorted,
            noise_coef=0.05, 
            guidance_scale=60
        )

        # --- B. 物理引导 (核心修正) ---
        reflection_result = apply_cylindrical_transform(img_distorted, ANAMORPHIC_GRID)
        
        # 使用 MSE Loss 对齐目标图片
        # 注意：这里对比的是 reflection_result 和 secret_target_tensor
        loss_secret = torch.nn.functional.mse_loss(reflection_result, secret_target_tensor) * GUIDANCE_STRENGTH
        
        loss_secret.backward()

        # --- C. 可视化 ---
        if iter_num % DISPLAY_INTERVAL == 0:
            with torch.no_grad():
                clear_output(wait=True)
                
                distorted_np = rp.as_numpy_image(img_distorted[0])
                reflected_np = rp.as_numpy_image(reflection_result[0])
                target_np = rp.as_numpy_image(secret_target_tensor[0])
                
                # 拼接：纸面图 | 反射效果 | 目标原图 (用于对比是否接近)
                combined_view = np.hstack([distorted_np, reflected_np, target_np])
                
                print(f"Iteration {iter_num} / {NUM_ITER}")
                print(f"左：纸面图 | 中：当前反射效果 | 右：你的目标原图")
                rp.display_image(combined_view)

        optim.step()
        optim.zero_grad()

except KeyboardInterrupt:
    print("停止训练。")

In [ ]:
# 修改最终成果展示块，增加辅助线
print("==== 最终成果展示（带打印辅助线） ====")

final_distorted = get_distorted_image()

# 将 Tensor 转为 Numpy 方便画线
distorted_np = rp.as_numpy_image(final_distorted[0])

# 画一个红色的圈，提示杯子放哪里
import cv2
center = SIZE // 2
radius_px = int(center * R_MIN)
# 在拷贝上画圈，不要影响原图
display_img = distorted_np.copy()
cv2.circle(display_img, (center, center), radius_px, (1, 0, 0), 2) # 红色定位圈

print(f"打印指南：请确保打印出的红色圆圈直径与你的杯子直径一致！")
rp.display_image(display_img)